In [79]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [80]:
df = pd.read_csv("../Data/Food Consumption Survey (1).csv")

In [81]:
df.head()

,Timestamp,House Number:,Name,Phone Number:,Area/Locality:,Number of family members,Diet type,"If Non Veg, Which days do you prefer eating meat?",Select the vegetables that your household purchases or consumes regularly. (up to 6),How much kg of vegetables s required by your household in a week approximately,Select the fruits that your household purchases or consumes regularly. (up to 5),Select the food grains that your household purchases or consumes regularly.,"If you consume rice, which type of rice is consumed by your household?",What dairy products is consumed by your household ?,How much kg of rice is consumed by your household in a day (in kg),How much kg of rice is consumed by your household in a month,How many Liters of milk is consumed by your household in a week,Where does your household primarily buy groceries?,How often do you buy vegetables?,How often do you buy groceries?,Recommended by? \nPlease mention the name of the person who sent this form to you.
0,2026/06/05 10:05:52 AM GMT+5:30,201,p mahema sai,9739697840,Doddakallasandra,4,Non Veg,Wednesday;Friday;Sunday,Onion;Tomato;Carrot;Potato;Beans;Chilli,NaN,Banana;Apple;Orange;Grapes;Watermelon;Pomegranate,Rice;Millets;Ragi;Wheat,White rice;Basmati;Brown rice,Milk;Curd;Panner;Buttermilk;Ice Cream,0.5 kg - 1 kg,6-10 kg,5-7 lt,METRO,weekly once,weekly twice,NaN
1,2026/06/05 10:05:52 AM GMT+5:30,20,Namitha N Reddy,9108291205,Koramangala,5,Veg,NaN,Onion;Tomato;Carrot;Potato;Beans;Cabbage;Drums...,NaN,Banana;Apple;Orange;Papaya;Guava;kiwi,Rice;Millets;Ragi;Barley,White rice,Milk;Curd;Panner,>0.5 kg,6-10 kg,5-7 lt,Local kirana stores,once in two weeks,once in two weeks,NaN
2,2026/06/05 10:10:07 AM GMT+5:30,703,N,9876512349,Doddakallasandra,4,Veg,NaN,Tomato;Carrot;Potato;Beans;Chilli,NaN,Banana;Apple;Orange;Papaya;Guava,Rice;Jawar;Millets;Wheat,White rice;Sona Masuri;Idli/Dosa Rice;Brown rice,Milk;Curd;Panner;Buttermilk;Ice Cream,< 0.5 kg,6-10 kg,3-5 lt,Blinkit,weekly twice,weekly once,NaN
3,2026/06/05 10:13:25 AM GMT+5:30,251,Shobhika Santhosh,7022167063,HSR,3,Non Veg,Wednesday;Friday;Saturday;Sunday,Onion;Tomato;Carrot;Potato;Brinjal;Beans;Cabba...,NaN,Banana;Apple;Mango,Rice;Millets;Ragi;Wheat,Red rice/Rajamudi,Milk;Curd;Panner;Buttermilk;Ice Cream,< 0.5 kg,3-6 kg,1-3 lt,LuLu Daily,weekly once,once in two weeks,NaN
4,2026/06/05 10:14:43 AM GMT+5:30,147,NaN,9513144130,Vijayanagar,4,Eggetarian,NaN,Onion;Tomato;Carrot;Beans;Cabbage,NaN,Banana;Apple;Orange;Papaya,Rice;Millets;Wheat;Maize,White rice;Sona Masuri;Idli/Dosa Rice,Milk;Curd,< 0.5 kg,6-10 kg,> 7 lt,Amazon Fresh & Amazon Now,weekly twice,once in two weeks,NaN


In [82]:
print("="*60)
print("DATASET OVERVIEW")
print("="*60)

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

print("\n")

df.info()

DATASET OVERVIEW
Rows    : 122
Columns : 21


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 21 columns):
 #   Column                                                                                Non-Null Count  Dtype 
---  ------                                                                                --------------  ----- 
 0   Timestamp                                                                             122 non-null    object
 1   House Number:                                                                         122 non-null    object
 2   Name                                                                                  118 non-null    object
 3   Phone Number:                                                                         122 non-null    object
 4   Area/Locality:                                                                        122 non-null    object
 5   Number of family members                      

In [83]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage (%)": round(df.isnull().mean()*100,2)
})

missing.sort_values("Missing Values", ascending=False)

,Missing Values,Percentage (%)
Recommended by? \nPlease mention the name of the person who sent this form to you.,111,90.98
How much kg of vegetables s required by your household in a week approximately,111,90.98
"If Non Veg, Which days do you prefer eating meat?",42,34.43
Name,4,3.28
Number of family members,1,0.82
What dairy products is consumed by your household ?,0,0.00
How often do you buy groceries?,0,0.00
How often do you buy vegetables?,0,0.00
Where does your household primarily buy groceries?,0,0.00
How many Liters of milk is consumed by your household in a week,0,0.00


In [84]:
df.columns = [
    "timestamp",
    "house_number",
    "name",
    "phone_number",
    "area",
    "family_size",
    "diet_type",
    "meat_days",
    "vegetables",
    "vegetable_quantity_week",
    "fruits",
    "food_grains",
    "rice_type",
    "dairy_products",
    "rice_per_day",
    "rice_per_month",
    "milk_per_week",
    "grocery_source",
    "vegetable_frequency",
    "grocery_frequency",
    "recommended_by"
]

In [85]:
df.columns.tolist()

['timestamp',
 'house_number',
 'name',
 'phone_number',
 'area',
 'family_size',
 'diet_type',
 'meat_days',
 'vegetables',
 'vegetable_quantity_week',
 'fruits',
 'food_grains',
 'rice_type',
 'dairy_products',
 'rice_per_day',
 'rice_per_month',
 'milk_per_week',
 'grocery_source',
 'vegetable_frequency',
 'grocery_frequency',
 'recommended_by']

In [86]:
clean_df = df.copy()

In [87]:
def clean_text(value):

    if pd.isna(value):
        return np.nan

    value = str(value)

    # remove leading and trailing spaces
    value = value.strip()

    # convert to lowercase
    value = value.lower()

    # remove multiple spaces
    value = re.sub(r"\s+", " ", value)

    return value
    

In [88]:
text_columns = [
    "name",
    "area",
    "diet_type",
    "meat_days",
    "vegetables",
    "fruits",
    "food_grains",
    "rice_type",
    "dairy_products",
    "grocery_source",
    "vegetable_frequency",
    "grocery_frequency",
    "recommended_by"
]

for column in text_columns:
    clean_df[column] = clean_df[column].apply(clean_text)

In [89]:
def clean_phone(phone):

    if pd.isna(phone):
        return np.nan

    phone = re.sub(r"\D", "", str(phone))

    return phone

In [90]:
clean_df.to_csv("../Data/cleaned_dataset (1).csv", index=False)

In [91]:
categorical_columns = [
    "area",
    "diet_type",
    "meat_days",
    "vegetables",
    "fruits",
    "food_grains",
    "rice_type",
    "dairy_products",
    "grocery_source",
    "vegetable_frequency",
    "grocery_frequency",
    "recommended_by"
]

In [92]:
for col in categorical_columns:
    print("=" * 80)
    print(f"COLUMN: {col.upper()}")
    print("=" * 80)
    print(clean_df[col].value_counts(dropna=False))
    print("\n\n")

COLUMN: AREA
area
vijayanagar                 20
doddakallasandra             8
sarjapur road                8
andrahalli                   7
rajajinagar                  6
nagarbhavi                   5
electronic city              3
vasanthnagar                 3
kengeri                      2
delhi                        2
srinagar                     2
isro layout                  2
jp nagar                     2
bannerghatta road            2
haveri                       1
rajarajeshwari nagar         1
vasanth nagar                1
padrayanapura                1
d group                      1
haralur                      1
kaggalipura                  1
arehalli                     1
g m palya                    1
nayandhalli                  1
basaweshwarnagar             1
chandra layout               1
bannerghatta                 1
konanakunte cross            1
d group employees layout     1
jp nagar 8th phase           1
banashankari 2nd stage       1
malleswaram          

In [93]:
#diet_pref cleaning

In [94]:
clean_df["diet_type"].unique()

array(['non veg', 'veg', 'eggetarian', 'jain', 'pescatarian', 'vegan'],
      dtype=object)

In [95]:
diet_mapping = {
    "veg": "Vegetarian",
    "vegetarian": "Vegetarian",
    "pure veg": "Vegetarian",
    "pure vegetarian": "Vegetarian",

    "non veg": "Non_Vegetarian",
    "non-veg": "Non_Vegetarian",
    "non vegetarian": "Non_Vegetarian",
    "nonvegetarian": "Non_Vegetarian",

    "eggetarian": "Eggetarian",
    "eggitarian": "Eggetarian",

    "vegan" : "Vegan",

    "pescatarian": "Pescatarian"
}

In [96]:
clean_df["diet_type"] = clean_df["diet_type"].replace(diet_mapping)

In [97]:
clean_df["diet_type"].value_counts()

diet_type
Non_Vegetarian    73
Vegetarian        29
Eggetarian        10
jain               4
Pescatarian        4
Vegan              2
Name: count, dtype: int64

In [98]:
#Grocery_store cleaning

In [99]:
clean_df["grocery_source"].unique()


array(['metro', 'local kirana stores', 'blinkit', 'lulu daily',
       'amazon fresh & amazon now', 'd mart', 'dmart ready', 'zepto',
       'shoprite mall', 'more mega store', 'flipkart minutes', 'reliance',
       'big basket', 'swiggy instamart', 'from our coffee estate',
       'k r market', 'all of this', 'general stores',
       'local kirana stores;reliance',
       'local kirana stores;reliance;d mart',
       'local kirana stores;k r market;blinkit;dmart ready',
       'local kirana stores;reliance;jiomart',
       'lulu daily;d mart;swiggy instamart',
       'k r market;d mart;flipkart minutes',
       'local kirana stores;metro;d mart',
       'local kirana stores;metro;d mart;zepto',
       'd mart;blinkit;swiggy instamart;zepto',
       'blinkit;swiggy instamart;jiomart;zepto',
       'local kirana stores;metro;reliance;d mart;blinkit',
       'blinkit;swiggy instamart;zepto', 'd mart;blinkit',
       'local kirana stores;k r market;gandhi bazaar;d mart;blinkit;swiggy inst

In [100]:
grocery_mapping = {
    "d mart": "Dmart",
    "dmart": "Dmart",
    "dmart ready":"Dmart Ready",
    
    "reliance": "Reliance fresh",
    "reliance fresh": "Reliance fresh",

    "big basket": "bigbasket",
    "bigbasket": "bigbasket",

    "zepto": "Zepto",
    "blinkit": "Blinkit",
    
    "local shop": "local store",
    "nearby store": "local store",

    "jiomart": "Jio Mart",
    "jio mart" : "Jio Mart",
    "metro" : "Metro",

    "big basket": "BigBasket",
    "LULU" : "LULU Daily"
}

In [101]:
clean_df["grocery_source"].value_counts()

grocery_source
local kirana stores                                                                                    29
d mart                                                                                                 20
metro                                                                                                   9
zepto                                                                                                   8
amazon fresh & amazon now                                                                               5
blinkit                                                                                                 4
flipkart minutes                                                                                        3
k r market                                                                                              3
big basket                                                                                              3
reliance                       

In [102]:
def clean_store_names(cell):

    if pd.isna(cell):
        return np.nan

    stores = [x.strip().lower() for x in cell.split(";")]

    cleaned = []

    for store in stores:
        store = grocery_mapping.get(store, store)
        cleaned.append(store)

    return ";".join(sorted(set(cleaned)))

In [103]:
clean_df["grocery_source"] = clean_df["grocery_source"].apply(clean_store_names)

In [104]:
from collections import Counter

store_counter = Counter()

for stores in clean_df["grocery_source"].dropna():

    for store in stores.split(";"):

        store_counter[store.strip()] += 1

store_counts = (
    pd.DataFrame(store_counter.items(), columns=["Store", "Count"])
      .sort_values("Count", ascending=False)
)

store_counts

,Store,Count
1,local kirana stores,42
5,Dmart,34
7,Zepto,16
2,Blinkit,14
0,Metro,13
11,Reliance fresh,9
15,k r market,7
4,amazon fresh & amazon now,7
13,swiggy instamart,7
6,Dmart Ready,6


In [105]:
sorted(clean_df["area"].dropna().unique())

['anand nagar hebbal post',
 'andrahalli',
 'arehalli',
 'banashankari',
 'banashankari 2nd stage',
 'bannerghatta',
 'bannerghatta road',
 'basaweshwarnagar',
 'batrayanpura',
 'btm layout',
 'california',
 'chandhapura',
 'chandra layout',
 'd group',
 'd group employees layout',
 'd group vidyamanyanagar',
 'delhi',
 'doddakallasandra',
 'e-city',
 'ecity',
 'electronic city',
 'electronic city-1',
 'g m palya',
 'hanumanthnagar',
 'haralur',
 'haveri',
 'hsr',
 'isro layout',
 'j p nagar',
 'jayanagar',
 'jp nagar',
 'jp nagar 8th phase',
 'kaggalipura',
 'kaglipura',
 'kengeri',
 'konanakunte cross',
 'koramangala',
 'kormangala',
 'kundapura',
 'malleswaram',
 'marathahalli',
 'nagadevanahalli',
 'nagarabhavi',
 'nagarbavi',
 'nagarbhavi',
 'nayandhalli',
 'padrayanapura',
 'r r nagar',
 'r t nagar',
 'rajajinagar',
 'rajarajeshwari nagar',
 'rv college',
 'sahakarnagar',
 'sarjapur road',
 'singasandra',
 'srinagar',
 'sunkadkatte',
 'tavarekere',
 'tripura',
 'tumkur',
 'uttara

In [106]:
clean_df["area"] = (
    clean_df["area"]
    .astype(str)
    .str.strip()
    .str.lower()
)

In [107]:
def normalize_area(area):

    if pd.isna(area):
        return area

    area = area.strip().lower()

    area_mapping = {

    "hsr":"hsr",
    "hsr layout":"hsr",

    "koramangla":"koramangala",

    "white field":"whitefield",

    "btm":"btm",
    "btm layout":"btm",

    "jayanagar":"jayanagar",

    "california":"vijayanagar",
    "kundapura":"vijayanagar",
    "delhi":"vijayanagar",
    "tripura":"vijayanagar",
    "tumkur":"vijayanagar",
        "haveri":"vijayanagar",

    "d group":"andrahalli",
    "d group employees layout":"andrahalli",
    "d group vidyamanyanagar":"andrahalli",

    "e-city":"electronic city",
    "ecity":"electronic city",
    "electronic city":"electronic city",
    "electronic city-1":"electronic city",

    "nagarabhavi":"nagarbhavi",
    "nagarbavi":"nagarbhavi",
    "nagarbhavi":"nagarbhavi",

    "vasanth nagar":"vasanthanagar",

    "rv college":"kengeri",
    "banashankari 2nd stage":"banashankari"
}


    return area_mapping.get(area, area)

clean_df["area"] = clean_df["area"].apply(normalize_area)

In [108]:
print("Unique Areas:", clean_df["area"].nunique())

sorted(clean_df["area"].unique())

Unique Areas: 48


['anand nagar hebbal post',
 'andrahalli',
 'arehalli',
 'banashankari',
 'bannerghatta',
 'bannerghatta road',
 'basaweshwarnagar',
 'batrayanpura',
 'btm',
 'chandhapura',
 'chandra layout',
 'doddakallasandra',
 'electronic city',
 'g m palya',
 'hanumanthnagar',
 'haralur',
 'hsr',
 'isro layout',
 'j p nagar',
 'jayanagar',
 'jp nagar',
 'jp nagar 8th phase',
 'kaggalipura',
 'kaglipura',
 'kengeri',
 'konanakunte cross',
 'koramangala',
 'kormangala',
 'malleswaram',
 'marathahalli',
 'nagadevanahalli',
 'nagarbhavi',
 'nayandhalli',
 'padrayanapura',
 'r r nagar',
 'r t nagar',
 'rajajinagar',
 'rajarajeshwari nagar',
 'sahakarnagar',
 'sarjapur road',
 'singasandra',
 'srinagar',
 'sunkadkatte',
 'tavarekere',
 'uttarahalli',
 'vasanthanagar',
 'vasanthnagar',
 'vijayanagar']

In [109]:
print(f"Unique Areas: {clean_df['area'].nunique()}")

Unique Areas: 48


In [110]:
# Vegetarian / Vegan households
clean_df.loc[
    (clean_df["diet_type"].isin(["vegetarian", "vegan"])) &
    (clean_df["meat_days"].isna()),
    "meat_days"
] = "not_applicable"

# Non-vegetarian households with missing values
clean_df.loc[
    (clean_df["diet_type"] == "non_vegetarian") &
    (clean_df["meat_days"].isna()),
    "meat_days"
] = "unknown"

In [111]:
day_order = {
    "monday": 1,
    "tuesday": 2,
    "wednesday": 3,
    "thursday": 4,
    "friday": 5,
    "saturday": 6,
    "sunday": 7
}

def clean_meat_days(cell):

    if pd.isna(cell):
        return cell

    # Keep these as they are
    if cell in ["not_applicable", "unknown"]:
        return cell

    days = [day.strip().lower() for day in str(cell).split(";")]

    # Remove duplicate days
    days = list(set(days))

    # Sort in week order
    days = sorted(days, key=lambda x: day_order.get(x, 99))

    return ";".join(days)

clean_df["meat_days"] = clean_df["meat_days"].apply(clean_meat_days)

In [112]:
from collections import Counter

day_counter = Counter()

for entry in clean_df["meat_days"]:

    if pd.isna(entry):
        continue

    if entry in ["not_applicable", "unknown"]:
        continue

    for day in entry.split(";"):
        day_counter[day.strip()] += 1

day_counts = (
    pd.DataFrame(day_counter.items(), columns=["Day", "Count"])
      .sort_values("Count", ascending=False)
)

day_counts

,Day,Count
2,sunday,78
0,wednesday,70
1,friday,46
4,tuesday,37
6,thursday,31
3,saturday,23
5,monday,23


In [113]:
from collections import Counter

vegetable_counter = Counter()

for row in clean_df["vegetables"].dropna():
    vegetables = [v.strip().lower() for v in row.split(";")]
    vegetable_counter.update(vegetables)

vegetable_df = (
    pd.DataFrame(
        vegetable_counter.items(),
        columns=["Vegetable", "Count"]
    )
    .sort_values("Count", ascending=False)
)

vegetable_df

,Vegetable,Count
1,tomato,114
0,onion,110
3,potato,103
2,carrot,96
5,chilli,86
4,beans,83
7,drumstick,54
6,cabbage,45
8,brinjal,45
9,cauliflower,40


In [114]:
for col in clean_df.columns:
    print(col)

timestamp
house_number
name
phone_number
area
family_size
diet_type
meat_days
vegetables
vegetable_quantity_week
fruits
food_grains
rice_type
dairy_products
rice_per_day
rice_per_month
milk_per_week
grocery_source
vegetable_frequency
grocery_frequency
recommended_by


In [115]:
freq_map = {

    "weekly": "Weekly",
    "every week": "Weekly",
    "once a week": "Weekly",

    "twice a week": "Twice a Week",

    "biweekly": "Biweekly",
    "once in two weeks": "Biweekly",
    "once every two weeks": "Biweekly",

    "monthly": "Monthly",
    "once a month": "Monthly",
    "as per requirements. mostly monthly":"Monthly",
    "1 month":"Monthly",
    "monthly once": "Monthly",
    "every 10 minutes":"Monthly",                        
    "once in a month":"Monthly" ,                       
     "5months once": "Monthly",
    "whne needed":"Monthly",
    "weekly twice":"Monthly"

}

clean_df["grocery_frequency"] = (

    clean_df["grocery_frequency"]
    .str.strip()
    .str.lower()
    .replace(freq_map)

)

In [116]:
clean_df["grocery_frequency"].value_counts(dropna=False)

grocery_frequency
Biweekly       51
Monthly        48
weekly once    22
when needed     1
Name: count, dtype: int64

In [117]:
veg_freq_map = {

    "daily": "Daily",

    "weekly": "Weekly",
    "once a week": "Weekly",

    "twice a week": "Twice a Week",

    "biweekly": "Biweekly",
    "once in two weeks":"Biweekly",

    "monthly": "Monthly",

    "once in two days": "Once in two days",       
   "once in 2 days"    : "Once in two days",       
   "every leap year"    : "Once in two days",
   "once every two days": "Once in two days"

}

clean_df["vegetable_frequency"] = (

    clean_df["vegetable_frequency"]
    .str.strip()
    .str.lower()
    .replace(veg_freq_map)

)

In [118]:
clean_df["vegetable_frequency"].value_counts(dropna=False)

vegetable_frequency
weekly twice        66
weekly once         42
Biweekly             8
Once in two days     4
Daily                1
weekly thrice        1
Name: count, dtype: int64

In [119]:
numeric_columns = [
    "family_size",
    "rice_per_day",
    "rice_per_month",
    "milk_per_week"
]

clean_df[numeric_columns].info()

clean_df[numeric_columns].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   family_size     121 non-null    object
 1   rice_per_day    122 non-null    object
 2   rice_per_month  122 non-null    object
 3   milk_per_week   122 non-null    object
dtypes: object(4)
memory usage: 3.9+ KB


,family_size,rice_per_day,rice_per_month,milk_per_week
0,4,0.5 kg - 1 kg,6-10 kg,5-7 lt
1,5,>0.5 kg,6-10 kg,5-7 lt
2,4,< 0.5 kg,6-10 kg,3-5 lt
3,3,< 0.5 kg,3-6 kg,1-3 lt
4,4,< 0.5 kg,6-10 kg,> 7 lt


In [120]:
for col in numeric_columns:
    print("="*50)
    print(col)
    print(clean_df[col].unique())

family_size
['4' '5' '3' '2' '6' '6+' '1' nan]
rice_per_day
['0.5 kg - 1 kg' '>0.5 kg' '< 0.5 kg' '1 kg - 2 kg' 'More than 2 kg'
 '1.5kg' 'O.4' '3kg' '.2' '.5' '2 kg' '500 g' '1 kg' '0.5kg' '1kg' '0.5'
 'HALF KG' '1/2 kg' '200 grams' '2' '0.250' '1' '(0.5-1) kg' '0.75'
 '0.5–0.8 kg per day' '15kgs']
rice_per_month
['6-10 kg' '3-6 kg' '10-14 kg' '18-22 kg' '14-18 kg' '>22 kg']
milk_per_week
['5-7 lt' '3-5 lt' '1-3 lt' '> 7 lt' '< 1lt']


In [121]:
for col in numeric_columns:
    print(f"\n------ {col} ------")

    for value in clean_df[col].dropna().unique():
        try:
            float(str(value))
        except:
            print(value)


------ family_size ------
6+

------ rice_per_day ------
0.5 kg - 1 kg
>0.5 kg
< 0.5 kg
1 kg - 2 kg
More than 2 kg
1.5kg
O.4
3kg
2 kg
500 g
1 kg
0.5kg
1kg
HALF KG
1/2 kg
200 grams
(0.5-1) kg
0.5–0.8 kg per day
15kgs

------ rice_per_month ------
6-10 kg
3-6 kg
10-14 kg
18-22 kg
14-18 kg
>22 kg

------ milk_per_week ------
5-7 lt
3-5 lt
1-3 lt
> 7 lt
< 1lt


In [122]:
import re
import numpy as np

def clean_numeric(value):

    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    # Extract the FIRST valid number
    match = re.search(r"\d+(\.\d+)?", value)

    if match:
        return float(match.group())

    return np.nan

In [123]:
for col in numeric_columns:
    clean_df[col] = clean_df[col].apply(clean_numeric)

In [124]:
for col in numeric_columns:
    invalid = clean_df[
        clean_df[col].astype(str).str.contains(r"[/-]", regex=True, na=False)
    ][col]

    if len(invalid):
        print(f"\n{col}")
        print(invalid)

In [125]:
clean_df[numeric_columns].info()

clean_df[numeric_columns].describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122 entries, 0 to 121
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   family_size     121 non-null    float64
 1   rice_per_day    121 non-null    float64
 2   rice_per_month  122 non-null    float64
 3   milk_per_week   122 non-null    float64
dtypes: float64(4)
memory usage: 3.9 KB


,family_size,rice_per_day,rice_per_month,milk_per_week
count,121.000000,121.000000,122.000000,122.000000
mean,4.099174,6.706612,10.270492,3.606557
std,1.325424,48.728325,5.580422,2.180141
min,1.000000,0.250000,3.000000,1.000000
25%,3.000000,0.500000,6.000000,1.000000
50%,4.000000,0.500000,10.000000,3.000000
75%,5.000000,1.000000,14.000000,5.000000
max,6.000000,500.000000,22.000000,7.000000


In [126]:
for col in numeric_columns:

    print("="*60)
    print(col)

    print("Minimum :", clean_df[col].min())
    print("Maximum :", clean_df[col].max())

family_size
Minimum : 1.0
Maximum : 6.0
rice_per_day
Minimum : 0.25
Maximum : 500.0
rice_per_month
Minimum : 3.0
Maximum : 22.0
milk_per_week
Minimum : 1.0
Maximum : 7.0


In [127]:
clean_df.loc[
    clean_df["rice_per_day"] == clean_df["rice_per_day"].max()
]

,timestamp,house_number,name,phone_number,area,family_size,diet_type,meat_days,vegetables,vegetable_quantity_week,fruits,food_grains,rice_type,dairy_products,rice_per_day,rice_per_month,milk_per_week,grocery_source,vegetable_frequency,grocery_frequency,recommended_by
96,2026/06/06 12:56:06 PM GMT+5:30,32,sanjana k,7975826787,konanakunte cross,4.0,Eggetarian,NaN,onion;tomato;carrot;beans;chilli,NaN,banana;apple;pomegranate,rice;ragi;wheat,white rice;red rice/rajamudi;basmati;idli/dosa...,milk;curd;panner;buttermilk;ice cream,500.0,10.0,5.0,Jio Mart;Reliance fresh;local kirana stores,weekly once,Monthly,NaN


In [128]:
clean_df.loc[
    clean_df["rice_per_day"] == 500,
    "rice_per_day"
] = 0.5

In [129]:
clean_df["rice_per_day"].describe()

count    121.000000
mean       2.578512
std       18.159508
min        0.250000
25%        0.500000
50%        0.500000
75%        1.000000
max      200.000000
Name: rice_per_day, dtype: float64

In [130]:
print(clean_df[clean_df["family_size"] > 10])

print(clean_df[clean_df["rice_per_day"] > 5])

print(clean_df[clean_df["rice_per_month"] > 30])

print(clean_df[clean_df["milk_per_week"] > 15])

Empty DataFrame
Columns: [timestamp, house_number, name, phone_number, area, family_size, diet_type, meat_days, vegetables, vegetable_quantity_week, fruits, food_grains, rice_type, dairy_products, rice_per_day, rice_per_month, milk_per_week, grocery_source, vegetable_frequency, grocery_frequency, recommended_by]
Index: []
                           timestamp                          house_number  \
107   2026/06/06 8:29:41 PM GMT+5:30  B605, Renaissance park 3 apartments    
119  2026/06/23 12:45:59 PM GMT+5:30                                   #52   

                     name phone_number         area  family_size  \
107  sharanya c bharadwaj   9380104818  malleswaram          4.0   
119            manyatha.r   6361718489   andrahalli          4.0   

          diet_type                          meat_days  \
107      Vegetarian                                NaN   
119  Non_Vegetarian  tuesday;wednesday;thursday;sunday   

                                          vegetables vegetabl

In [131]:
clean_df.loc[107, "rice_per_day"] = 0.2

clean_df.loc[119, "rice_per_day"] = 1.5

In [132]:
clean_df["rice_per_day"].describe()

count    121.000000
mean       0.815702
std        0.782119
min        0.200000
25%        0.500000
50%        0.500000
75%        1.000000
max        5.000000
Name: rice_per_day, dtype: float64

In [133]:
clean_df[clean_df["rice_per_day"] > 5]

,timestamp,house_number,name,phone_number,area,family_size,diet_type,meat_days,vegetables,vegetable_quantity_week,fruits,food_grains,rice_type,dairy_products,rice_per_day,rice_per_month,milk_per_week,grocery_source,vegetable_frequency,grocery_frequency,recommended_by


In [134]:
clean_df.columns.tolist()

['timestamp',
 'house_number',
 'name',
 'phone_number',
 'area',
 'family_size',
 'diet_type',
 'meat_days',
 'vegetables',
 'vegetable_quantity_week',
 'fruits',
 'food_grains',
 'rice_type',
 'dairy_products',
 'rice_per_day',
 'rice_per_month',
 'milk_per_week',
 'grocery_source',
 'vegetable_frequency',
 'grocery_frequency',
 'recommended_by']

In [135]:
columns_to_drop = [
    "timestamp",
    "vegetable_quantity_week",
    "recommended_by"
]

clean_df.drop(
    columns=columns_to_drop,
    errors="ignore",      
    inplace=True
)

print("Remaining Columns:")
print(clean_df.columns.tolist())

Remaining Columns:
['house_number', 'name', 'phone_number', 'area', 'family_size', 'diet_type', 'meat_days', 'vegetables', 'fruits', 'food_grains', 'rice_type', 'dairy_products', 'rice_per_day', 'rice_per_month', 'milk_per_week', 'grocery_source', 'vegetable_frequency', 'grocery_frequency']


In [136]:
clean_df.head()

,house_number,name,phone_number,area,family_size,diet_type,meat_days,vegetables,fruits,food_grains,rice_type,dairy_products,rice_per_day,rice_per_month,milk_per_week,grocery_source,vegetable_frequency,grocery_frequency
0,201,p mahema sai,9739697840,doddakallasandra,4.0,Non_Vegetarian,wednesday;friday;sunday,onion;tomato;carrot;potato;beans;chilli,banana;apple;orange;grapes;watermelon;pomegranate,rice;millets;ragi;wheat,white rice;basmati;brown rice,milk;curd;panner;buttermilk;ice cream,0.5,6.0,5.0,Metro,weekly once,Monthly
1,20,namitha n reddy,9108291205,koramangala,5.0,Vegetarian,NaN,onion;tomato;carrot;potato;beans;cabbage;drums...,banana;apple;orange;papaya;guava;kiwi,rice;millets;ragi;barley,white rice,milk;curd;panner,0.5,6.0,5.0,local kirana stores,Biweekly,Biweekly
2,703,n,9876512349,doddakallasandra,4.0,Vegetarian,NaN,tomato;carrot;potato;beans;chilli,banana;apple;orange;papaya;guava,rice;jawar;millets;wheat,white rice;sona masuri;idli/dosa rice;brown rice,milk;curd;panner;buttermilk;ice cream,0.5,6.0,3.0,Blinkit,weekly twice,weekly once
3,251,shobhika santhosh,7022167063,hsr,3.0,Non_Vegetarian,wednesday;friday;saturday;sunday,onion;tomato;carrot;potato;brinjal;beans;cabba...,banana;apple;mango,rice;millets;ragi;wheat,red rice/rajamudi,milk;curd;panner;buttermilk;ice cream,0.5,3.0,1.0,lulu daily,weekly once,Biweekly
4,147,NaN,9513144130,vijayanagar,4.0,Eggetarian,NaN,onion;tomato;carrot;beans;cabbage,banana;apple;orange;papaya,rice;millets;wheat;maize,white rice;sona masuri;idli/dosa rice,milk;curd,0.5,6.0,7.0,amazon fresh & amazon now,weekly twice,Biweekly


In [137]:

print(clean_df.shape)

print(clean_df.isnull().sum())

print("Duplicate rows:", clean_df.duplicated().sum())


print(clean_df.dtypes)


clean_df.head()

(122, 18)
house_number            0
name                    4
phone_number            0
area                    0
family_size             1
diet_type               0
meat_days              42
vegetables              0
fruits                  0
food_grains             0
rice_type               0
dairy_products          0
rice_per_day            1
rice_per_month          0
milk_per_week           0
grocery_source          0
vegetable_frequency     0
grocery_frequency       0
dtype: int64
Duplicate rows: 0
house_number            object
name                    object
phone_number            object
area                    object
family_size            float64
diet_type               object
meat_days               object
vegetables              object
fruits                  object
food_grains             object
rice_type               object
dairy_products          object
rice_per_day           float64
rice_per_month         float64
milk_per_week          float64
grocery_source          ob

,house_number,name,phone_number,area,family_size,diet_type,meat_days,vegetables,fruits,food_grains,rice_type,dairy_products,rice_per_day,rice_per_month,milk_per_week,grocery_source,vegetable_frequency,grocery_frequency
0,201,p mahema sai,9739697840,doddakallasandra,4.0,Non_Vegetarian,wednesday;friday;sunday,onion;tomato;carrot;potato;beans;chilli,banana;apple;orange;grapes;watermelon;pomegranate,rice;millets;ragi;wheat,white rice;basmati;brown rice,milk;curd;panner;buttermilk;ice cream,0.5,6.0,5.0,Metro,weekly once,Monthly
1,20,namitha n reddy,9108291205,koramangala,5.0,Vegetarian,NaN,onion;tomato;carrot;potato;beans;cabbage;drums...,banana;apple;orange;papaya;guava;kiwi,rice;millets;ragi;barley,white rice,milk;curd;panner,0.5,6.0,5.0,local kirana stores,Biweekly,Biweekly
2,703,n,9876512349,doddakallasandra,4.0,Vegetarian,NaN,tomato;carrot;potato;beans;chilli,banana;apple;orange;papaya;guava,rice;jawar;millets;wheat,white rice;sona masuri;idli/dosa rice;brown rice,milk;curd;panner;buttermilk;ice cream,0.5,6.0,3.0,Blinkit,weekly twice,weekly once
3,251,shobhika santhosh,7022167063,hsr,3.0,Non_Vegetarian,wednesday;friday;saturday;sunday,onion;tomato;carrot;potato;brinjal;beans;cabba...,banana;apple;mango,rice;millets;ragi;wheat,red rice/rajamudi,milk;curd;panner;buttermilk;ice cream,0.5,3.0,1.0,lulu daily,weekly once,Biweekly
4,147,NaN,9513144130,vijayanagar,4.0,Eggetarian,NaN,onion;tomato;carrot;beans;cabbage,banana;apple;orange;papaya,rice;millets;wheat;maize,white rice;sona masuri;idli/dosa rice,milk;curd,0.5,6.0,7.0,amazon fresh & amazon now,weekly twice,Biweekly


In [138]:
clean_df.loc[
    clean_df["diet_type"].isin(["Vegetarian", "Eggetarian"]),
    "meat_days"
] = clean_df.loc[
    clean_df["diet_type"].isin(["Vegetarian", "Eggetarian"]),
    "meat_days"
].fillna("not_applicable")

In [139]:
clean_df.loc[
    (clean_df["diet_type"] == "Non_Vegetarian") &
    (clean_df["meat_days"].isna()),
    "meat_days"
] = "unknown"

In [140]:
clean_df["rice_per_day"].describe()

count    121.000000
mean       0.815702
std        0.782119
min        0.200000
25%        0.500000
50%        0.500000
75%        1.000000
max        5.000000
Name: rice_per_day, dtype: float64

In [141]:
clean_df["phone_number"].duplicated().sum()

3

In [142]:
print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)

print(f"Rows    : {clean_df.shape[0]}")
print(f"Columns : {clean_df.shape[1]}")

print("\nMissing Values\n")
print(clean_df.isnull().sum())

print("\nDuplicate Rows :", clean_df.duplicated().sum())

print("\nData Types\n")
print(clean_df.dtypes)

print("\nDataset Preview\n")
display(clean_df.head())

FINAL DATASET SUMMARY
Rows    : 122
Columns : 18

Missing Values

house_number           0
name                   4
phone_number           0
area                   0
family_size            1
diet_type              0
meat_days              7
vegetables             0
fruits                 0
food_grains            0
rice_type              0
dairy_products         0
rice_per_day           1
rice_per_month         0
milk_per_week          0
grocery_source         0
vegetable_frequency    0
grocery_frequency      0
dtype: int64

Duplicate Rows : 0

Data Types

house_number            object
name                    object
phone_number            object
area                    object
family_size            float64
diet_type               object
meat_days               object
vegetables              object
fruits                  object
food_grains             object
rice_type               object
dairy_products          object
rice_per_day           float64
rice_per_month         float64
milk

,house_number,name,phone_number,area,family_size,diet_type,meat_days,vegetables,fruits,food_grains,rice_type,dairy_products,rice_per_day,rice_per_month,milk_per_week,grocery_source,vegetable_frequency,grocery_frequency
0,201,p mahema sai,9739697840,doddakallasandra,4.0,Non_Vegetarian,wednesday;friday;sunday,onion;tomato;carrot;potato;beans;chilli,banana;apple;orange;grapes;watermelon;pomegranate,rice;millets;ragi;wheat,white rice;basmati;brown rice,milk;curd;panner;buttermilk;ice cream,0.5,6.0,5.0,Metro,weekly once,Monthly
1,20,namitha n reddy,9108291205,koramangala,5.0,Vegetarian,not_applicable,onion;tomato;carrot;potato;beans;cabbage;drums...,banana;apple;orange;papaya;guava;kiwi,rice;millets;ragi;barley,white rice,milk;curd;panner,0.5,6.0,5.0,local kirana stores,Biweekly,Biweekly
2,703,n,9876512349,doddakallasandra,4.0,Vegetarian,not_applicable,tomato;carrot;potato;beans;chilli,banana;apple;orange;papaya;guava,rice;jawar;millets;wheat,white rice;sona masuri;idli/dosa rice;brown rice,milk;curd;panner;buttermilk;ice cream,0.5,6.0,3.0,Blinkit,weekly twice,weekly once
3,251,shobhika santhosh,7022167063,hsr,3.0,Non_Vegetarian,wednesday;friday;saturday;sunday,onion;tomato;carrot;potato;brinjal;beans;cabba...,banana;apple;mango,rice;millets;ragi;wheat,red rice/rajamudi,milk;curd;panner;buttermilk;ice cream,0.5,3.0,1.0,lulu daily,weekly once,Biweekly
4,147,NaN,9513144130,vijayanagar,4.0,Eggetarian,not_applicable,onion;tomato;carrot;beans;cabbage,banana;apple;orange;papaya,rice;millets;wheat;maize,white rice;sona masuri;idli/dosa rice,milk;curd,0.5,6.0,7.0,amazon fresh & amazon now,weekly twice,Biweekly


In [143]:
target_cols = [
    "rice_per_day",
    "rice_per_month",
    "milk_per_week"
]

for col in target_cols:
    print("\n", col)
    print(clean_df[col].unique()[:20])


 rice_per_day
[0.5  1.   2.   1.5  4.   3.   5.    nan 0.2  0.25 0.75]

 rice_per_month
[ 6.  3. 10. 18. 14. 22.]

 milk_per_week
[5. 3. 1. 7.]


In [144]:
import re
import numpy as np

def convert_to_numeric(value):

    if pd.isna(value):
        return np.nan

    value = str(value).lower().strip()

    value = value.replace("kgs","")
    value = value.replace("kg","")
    value = value.replace("litres","")
    value = value.replace("liters","")
    value = value.replace("litre","")
    value = value.replace("liter","")
    value = value.replace("ltrs","")
    value = value.replace("ltr","")
    value = value.replace("lt","")

    value = value.strip()

    
    if value.startswith("<"):
        try:
            return float(value.replace("<",""))
        except:
            return np.nan

    if "-" in value:
        try:
            nums = value.split("-")
            nums = [float(i.strip()) for i in nums]
            return sum(nums)/len(nums)
        except:
            return np.nan

    num = re.findall(r"\d+\.?\d*", value)

    if len(num):
        return float(num[0])

    return np.nan

In [145]:
clean_df["rice_per_day"] = (
    clean_df["rice_per_day"]
    .apply(convert_to_numeric)
)

clean_df["rice_per_month"] = (
    clean_df["rice_per_month"]
    .apply(convert_to_numeric)
)

clean_df["milk_per_week"] = (
    clean_df["milk_per_week"]
    .apply(convert_to_numeric)
)

In [146]:
clean_df[target_cols].head(15)

,rice_per_day,rice_per_month,milk_per_week
0,0.5,6.0,5.0
1,0.5,6.0,5.0
2,0.5,6.0,3.0
3,0.5,3.0,1.0
4,0.5,6.0,7.0
5,1.0,10.0,7.0
6,0.5,10.0,1.0
7,0.5,3.0,1.0
8,0.5,18.0,3.0
9,0.5,14.0,7.0


In [147]:
clean_df.to_csv(
    "../Data/cleaned_dataset.csv",
    index=False
)

print("cleaned_dataset.csv saved successfully!")

cleaned_dataset.csv saved successfully!


In [148]:
print(clean_df.shape)
print(clean_df["area"].nunique())

(122, 18)
48


In [149]:
for col in ["vegetables", "fruits", "food_grains", "dairy_products", "grocery_source"]:
    print(f"\n{col}")
    print(clean_df[col].sample(10))


vegetables
3     onion;tomato;carrot;potato;brinjal;beans;cabba...
35              onion;tomato;carrot;potato;beans;chilli
44              onion;tomato;carrot;potato;beans;chilli
57       beans;cauliflower;okra;drumstick;bottle gourd,
71         onion;tomato;carrot;potato;brinjal;drumstick
39      onion;tomato;carrot;potato;brinjal;beans;chilli
54    onion;tomato;carrot;okra;drumstick;chilli;cucu...
0               onion;tomato;carrot;potato;beans;chilli
59         onion;tomato;carrot;potato;beans;cauliflower
74         onion;tomato;carrot;beans;cauliflower;chilli
Name: vegetables, dtype: object

fruits
87                              orange;grapes;watermelon
118           banana;apple;orange;watermelon;pomegranate
28                        banana;apple;orange;watermelon
119                      banana;apple;papaya;guava;mango
63                  banana;apple;papaya;guava;blue berry
78                        banana;orange;watermelon;guava
106                            banana;apple;pa

In [150]:
def fix_separators(text):

    if pd.isna(text):
        return text

    text = str(text).lower()

    # Replace separators
    text = text.replace(",", ";")
    text = text.replace("&", ";")
    text = text.replace("/", ";")

    # Split, strip, remove blanks
    items = [i.strip() for i in text.split(";") if i.strip()]

    # Remove duplicates while preserving order
    items = list(dict.fromkeys(items))

    return ";".join(items)

clean_df["fruits"] = clean_df["fruits"].apply(fix_separators)
clean_df["grocery_source"] = clean_df["grocery_source"].apply(fix_separators)

In [151]:
print(clean_df["fruits"].sample(10))
print(clean_df["grocery_source"].sample(10))

115                      banana;apple;grapes;guava;mango
15                banana;apple;orange;papaya;pomegranate
89                 banana;apple;papaya;pomegranate;guava
78                        banana;orange;watermelon;guava
2                       banana;apple;orange;papaya;guava
36     banana;orange;papaya;watermelon;mangos during ...
6                  banana;apple;orange;papaya;watermelon
55     banana;apple;grapes;watermelon;pomegranate;plu...
88                banana;apple;orange;papaya;pomegranate
110                              banana;apple;watermelon
Name: fruits, dtype: object
27                      dmart
90                    blinkit
50                  bigbasket
76                      zepto
9                     blinkit
7     amazon fresh;amazon now
41                      zepto
2                     blinkit
70                 k r market
46                      dmart
Name: grocery_source, dtype: object


In [152]:
clean_df.to_csv("../Data/cleaned_dataset.csv", index=False)

In [153]:
print(clean_df.columns.tolist())

['house_number', 'name', 'phone_number', 'area', 'family_size', 'diet_type', 'meat_days', 'vegetables', 'fruits', 'food_grains', 'rice_type', 'dairy_products', 'rice_per_day', 'rice_per_month', 'milk_per_week', 'grocery_source', 'vegetable_frequency', 'grocery_frequency']


In [155]:
import pandas as pd

df = pd.read_csv("Data/cleaned_dataset.csv")

print(df[df["family_size"].isna()])

print(df[df["family_size"] == ""])

FileNotFoundError: [Errno 2] No such file or directory: 'Data/cleaned_dataset.csv'

In [156]:
import pandas as pd

df = pd.read_csv("../Data/cleaned_dataset.csv")

print(df[df["family_size"].isna()])

print(df[df["family_size"] == ""])

    house_number       name phone_number          area  family_size  \
116          110  pradyumna   9742460077  bannerghatta          NaN   

      diet_type       meat_days                                vegetables  \
116  Vegetarian  not_applicable  onion;tomato;carrot;potato;brinjal;beans   

                                fruits food_grains   rice_type dairy_products  \
116  banana;apple;orange;papaya;grapes        rice  white rice      ice cream   

     rice_per_day  rice_per_month  milk_per_week       grocery_source  \
116           1.0            10.0            1.0  local kirana stores   

    vegetable_frequency grocery_frequency  
116         weekly once       weekly once  
Empty DataFrame
Columns: [house_number, name, phone_number, area, family_size, diet_type, meat_days, vegetables, fruits, food_grains, rice_type, dairy_products, rice_per_day, rice_per_month, milk_per_week, grocery_source, vegetable_frequency, grocery_frequency]
Index: []


In [157]:
import pandas as pd

from utils.zone_mapping import ZONE_MAP

df = pd.read_csv("../Data/cleaned_dataset.csv")

unknown = []

for area in df["area"].dropna().unique():

    area = str(area).strip().lower()

    if area not in ZONE_MAP:
        unknown.append(area)

print("Unknown Areas:")
print(sorted(unknown))

ModuleNotFoundError: No module named 'utils'

In [159]:
import pandas as pd
import sys
sys.path.append("..")

from utils.zone_mapping import ZONE_MAP

from utils.zone_mapping import ZONE_MAP

df = pd.read_csv("../Data/cleaned_dataset.csv")

unknown = (
    df[
        ~df["area"]
        .str.strip()
        .str.lower()
        .isin(ZONE_MAP.keys())
    ]
    .groupby("area")
    .size()
    .sort_values(ascending=False)
)

print(unknown)

area
andrahalli            10
vasanthnagar           3
srinagar               2
batrayanpura           1
nagadevanahalli        1
tavarekere             1
r t nagar              1
r r nagar              1
padrayanapura          1
nayandhalli            1
malleswaram            1
chandhapura            1
basaweshwarnagar       1
konanakunte cross      1
jp nagar 8th phase     1
j p nagar              1
hanumanthnagar         1
g m palya              1
kormangala             1
dtype: int64


In [160]:
unknown = (
    df[
        ~df["area"]
        .str.strip()
        .str.lower()
        .isin(ZONE_MAP.keys())
    ]
    .groupby("area")
    .size()
    .sort_values(ascending=False)
)

print(unknown)

area
andrahalli            10
vasanthnagar           3
srinagar               2
batrayanpura           1
nagadevanahalli        1
tavarekere             1
r t nagar              1
r r nagar              1
padrayanapura          1
nayandhalli            1
malleswaram            1
chandhapura            1
basaweshwarnagar       1
konanakunte cross      1
jp nagar 8th phase     1
j p nagar              1
hanumanthnagar         1
g m palya              1
kormangala             1
dtype: int64


In [161]:
unknown = (
    df[
        ~df["area"]
        .str.strip()
        .str.lower()
        .isin(ZONE_MAP.keys())
    ]
    .groupby("area")
    .size()
    .sort_values(ascending=False)
)

print(unknown)

area
andrahalli            10
vasanthnagar           3
srinagar               2
batrayanpura           1
nagadevanahalli        1
tavarekere             1
r t nagar              1
r r nagar              1
padrayanapura          1
nayandhalli            1
malleswaram            1
chandhapura            1
basaweshwarnagar       1
konanakunte cross      1
jp nagar 8th phase     1
j p nagar              1
hanumanthnagar         1
g m palya              1
kormangala             1
dtype: int64
